## Cleaning the General Payments Dataset

This script takes in the raw file from the CMS_General_Payments_2021_2023_raw.csv.

Data Cleaning Steps:
- Change all data to columns to snake case
- Use the accompanying data dictionary to change the data types of specified columns
- filter dataset to remove Nas in total_amount_of_payment_us_dollars
- Remove numbers after the dash in the zipcode column
- Drop unrelated columns
- filter dataset to remove Nas in covered_recipient_npi
- Clean text columns
- Filter out all non-physician practitioners
- Backfill product names

Split the Data into Different CSV Dim Tables:
Split the general payments data into domain specific dimensions. Including manufacturing, provider info, products info, and then the general fact records table.

*Returns these CSVs*
- general_payments-sliced_clean.csv (Record level remaining fact data)
- general_payments_products_clean.csv (Product information, one product ID (PDI) per Row, keeping only most recent program year info)
- general_payments_providers_clean.csv (NPI level data and provider info, one NPI per Row, keeping only most recent program year info)
- general_payments_manufacturers_clean.csv (Record level remaining fact data)

**Return an intact and whole cleaned general payments dataset in chunked csvs. This data is before splitting into dims**

In [1]:
# Packages
import numpy as np
import pandas as pd
import glob

In [2]:
import re

def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

### Clean the General Payments Raw File

In [3]:
# Read in General Payments CSV from file path
df = pd.read_csv("/dsa/groups/casestudycf25/team02/CMS_General_Payments_2021_2023_raw.csv", dtype=str)

df = df.rename(columns={col: to_snake_case(col) for col in df.columns})

df.head(5)

,change_type,covered_recipient_type,teaching_hospital_ccn,teaching_hospital_id,teaching_hospital_name,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_first_name,covered_recipient_middle_name,covered_recipient_last_name,...,associated_drug_or_biological_ndc_4,associated_device_or_medical_supply_pdi_4,covered_or_noncovered_indicator_5,indicate_drug_or_biological_or_device_or_medical_supply_5,product_category_or_therapeutic_area_5,name_of_drug_or_biological_or_device_or_medical_supply_5,associated_drug_or_biological_ndc_5,associated_device_or_medical_supply_pdi_5,program_year,payment_publication_date
0,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
1,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
2,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
3,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
4,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,11322702,1932840923,CLAYTON,D,FOSTER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025


In [4]:
print(f"Original Dataset Shape: {df.shape}")

Original Dataset Shape: (7917764, 91)


In [5]:
df.columns

Index(['change_type', 'covered_recipient_type', 'teaching_hospital_ccn',
       'teaching_hospital_id', 'teaching_hospital_name',
       'covered_recipient_profile_id', 'covered_recipient_npi',
       'covered_recipient_first_name', 'covered_recipient_middle_name',
       'covered_recipient_last_name', 'covered_recipient_name_suffix',
       'recipient_primary_business_street_address_line1',
       'recipient_primary_business_street_address_line2', 'recipient_city',
       'recipient_state', 'recipient_zip_code', 'recipient_country',
       'recipient_province', 'recipient_postal_code',
       'covered_recipient_primary_type_1', 'covered_recipient_primary_type_2',
       'covered_recipient_primary_type_3', 'covered_recipient_primary_type_4',
       'covered_recipient_primary_type_5', 'covered_recipient_primary_type_6',
       'covered_recipient_specialty_1', 'covered_recipient_specialty_2',
       'covered_recipient_specialty_3', 'covered_recipient_specialty_4',
       'covered_recipie

In [6]:
# All other columns strings per data dictionary
# program_year -> integer
# payment_publication_date -> date
# date_of_payment -> date
# total_amount_of_payment_usdollars -> number
# number_of_payments_included_in_total_amount -> integer
# covered_recipient_npi -> integer

special_types = {
    "program_year": "Int64",   # Pandas nullable integer
    "number_of_payments_included_in_total_amount": "Int64",
    "covered_recipient_npi": "Int64",
    "total_amount_of_payment_us_dollars": "float64",
    "payment_publication_date": "datetime64[ns]",
    "date_of_payment": "datetime64[ns]",
}

In [7]:
for col, dtype in special_types.items():
    if dtype == "datetime64[ns]":
        df[col] = pd.to_datetime(df[col], errors = "coerce")

    elif dtype == "Int64":
        df[col] = pd.to_numeric(df[col], errors = "coerce").astype("Int64")

    elif dtype == "float64":
        df[col] = pd.to_numeric(df[col], errors = "coerce")

    else:
        df[col] = df[col].astype(dtype)

In [8]:
print(df.dtypes)

change_type                                                         object
covered_recipient_type                                              object
teaching_hospital_ccn                                               object
teaching_hospital_id                                                object
teaching_hospital_name                                              object
                                                                 ...      
name_of_drug_or_biological_or_device_or_medical_supply_5            object
associated_drug_or_biological_ndc_5                                 object
associated_device_or_medical_supply_pdi_5                           object
program_year                                                         Int64
payment_publication_date                                    datetime64[ns]
Length: 91, dtype: object


In [9]:
# Remove numbers after the dash in the zipcode column. Example 37212-4354 -> 37212
df["recipient_zip_code"] = df["recipient_zip_code"].str.replace(r"-\d{4}$", "", regex=True)

In [10]:
# Any column that starts with name_of is trimmed and uppercased
cols = [c for c in df.columns if c.startswith("name_of")]

df[cols] = df[cols].apply(lambda col: col.str.upper().str.strip())

In [11]:
# Any column that starts with product_category_ is trimmed and uppercased
cols = [c for c in df.columns if c.startswith("product_category")]

df[cols] = df[cols].apply(lambda col: col.str.upper().str.strip())

In [12]:
# Clean manufacturer name
df["applicable_manufacturer_or_applicable_gpo_making_payment_name"] = (
    df["applicable_manufacturer_or_applicable_gpo_making_payment_name"]
        .str.upper() # Uppercase
        .str.replace(r"[.,]", "", regex=True) # Remove periods
        .str.strip() # trim
)

In [13]:
# Check blank rows in manufacturer name
blank_rows = df[df["applicable_manufacturer_or_applicable_gpo_making_payment_name"]
                  .astype(str).str.strip().eq("")]
blank_rows

,change_type,covered_recipient_type,teaching_hospital_ccn,teaching_hospital_id,teaching_hospital_name,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_first_name,covered_recipient_middle_name,covered_recipient_last_name,...,associated_drug_or_biological_ndc_4,associated_device_or_medical_supply_pdi_4,covered_or_noncovered_indicator_5,indicate_drug_or_biological_or_device_or_medical_supply_5,product_category_or_therapeutic_area_5,name_of_drug_or_biological_or_device_or_medical_supply_5,associated_drug_or_biological_ndc_5,associated_device_or_medical_supply_pdi_5,program_year,payment_publication_date


In [14]:
###################
# Drop blank columns
#####################
df.dropna(axis=1, how="all", inplace=True)

In [15]:
#####################
# Drop unrelated columns
#####################
drops_cols = [
    'covered_recipient_first_name', 'covered_recipient_middle_name',  #unnecessary to have name
    'covered_recipient_last_name', 'covered_recipient_name_suffix',
    'teaching_hospital_ccn', 'teaching_hospital_id', 'teaching_hospital_name', #no teaching hospitals

    'recipient_province', #unnecessary location info
    
    'covered_recipient_primary_type_2','covered_recipient_primary_type_3', 'covered_recipient_specialty_2', # mostly blank
    'related_product_indicator','delay_in_publication_indicator', #only Nos so remove
     'associated_drug_or_biological_ndc_1', 'associated_drug_or_biological_ndc_2', # remove unrelated drug info
    'associated_drug_or_biological_ndc_3','associated_drug_or_biological_ndc_4', 'associated_drug_or_biological_ndc_5',
    'submitting_applicable_manufacturer_or_applicable_gpo_name' # mostly the same as applicable manufacturer name
    ,'contextual_information' # mostly blank
]

In [16]:
df.drop(columns = drops_cols, inplace = True) # drop blank columns

In [17]:
print(f"New Dataset Shape After Dropping Cols: {df.shape}")

New Dataset Shape After Dropping Cols: (7917764, 64)


In [18]:
df['dispute_status_for_publication'].value_counts()

No     7917276
Yes        488
Name: dispute_status_for_publication, dtype: int64

### Filter the General Payments Dataset

In [19]:
###################################
# filter dataset to remove Nas in total_amount_of_payment_us_dollars
###################################
df = df.dropna(subset=["total_amount_of_payment_us_dollars"])

In [20]:
###################################
# filter dataset to remove Nas in covered_recipient_npi
###################################
df = df.dropna(subset=["covered_recipient_npi"])

In [21]:
###################################
# Filter all non-physician practitioners
###################################
df = df[df["covered_recipient_type"] != "Covered Recipient Non-Physician Practitioner"]

In [22]:
### Update the wording for the nature of payments

mapping = {
    'Charitable Contribution': 'Charity',
    'Entertainment': 'Entertainment',
    'Debt forgiveness': 'Debt Forgiveness',
    'Current or prospective ownership or investment interest': 'Ownership or Investment',
    'Compensation for serving as faculty or as a speaker for a medical education program': 'Faculty or Speaker',
    'Education': 'Education',
    'Acquisitions': 'Acquisition',
    'Compensation for services other than consulting, including serving as faculty or as a speaker at a venue other than a continuing education program': 'Other Services',
    'Food and Beverage': 'Food and Beverage',
    'Consulting Fee': 'Consulting'
}

df['nature_short_descr'] = df['nature_of_payment_or_transfer_of_value'].map(mapping)


In [23]:
### Split the recipient specialty into hierarchy columns
splits = df['covered_recipient_specialty_1'].str.split('|', expand=True)
splits.columns = [f'specialty_lvl{i+1}' for i in range(splits.shape[1])]
df = pd.concat([df, splits], axis=1)

In [24]:
print("Check if there are any duplicate record ids in final dataset: ")
df["record_id"].duplicated().any()

Check if there are any duplicate record ids in final dataset: 


False

## CREATE DIMENSIONS TO REDUCE DATA LOAD

Split the general payments data into domain specific dimensions. Including manufacturing, provider info, products info, and then the general fact records table. 

########################################
# CREATE PROVIDERS DIM
########################################

In [25]:

df.columns

Index(['change_type', 'covered_recipient_type', 'covered_recipient_profile_id',
       'covered_recipient_npi',
       'recipient_primary_business_street_address_line1',
       'recipient_primary_business_street_address_line2', 'recipient_city',
       'recipient_state', 'recipient_zip_code', 'recipient_country',
       'recipient_postal_code', 'covered_recipient_primary_type_1',
       'covered_recipient_specialty_1',
       'covered_recipient_license_state_code1',
       'covered_recipient_license_state_code2',
       'covered_recipient_license_state_code3',
       'covered_recipient_license_state_code4',
       'covered_recipient_license_state_code5',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'applicable_manufacturer_or_applicable_gpo_making_payment_name',
       'applicable_manufacturer_or_applicable_gpo_making_payment_state',
       'applicable_manufacturer_or_applicable_gpo_making_payment_country',
       'total_amount_of_payment_us_dollars', 'd

In [26]:
providers = df[[
"covered_recipient_profile_id",
"covered_recipient_npi",
"covered_recipient_type",
"covered_recipient_primary_type_1",
"covered_recipient_specialty_1",
"specialty_lvl1",
"specialty_lvl2",
"specialty_lvl3",
"covered_recipient_license_state_code1",
"covered_recipient_license_state_code2",
"covered_recipient_license_state_code3",
"covered_recipient_license_state_code4",
"covered_recipient_license_state_code5",
'recipient_primary_business_street_address_line1',
'recipient_primary_business_street_address_line2', 'recipient_city',
'recipient_state', 'recipient_zip_code', 'recipient_country',
'recipient_postal_code', 
'program_year']]

In [27]:
providers.sort_values(by = "program_year", ascending = False)

,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_type,covered_recipient_primary_type_1,covered_recipient_specialty_1,specialty_lvl1,specialty_lvl2,specialty_lvl3,covered_recipient_license_state_code1,covered_recipient_license_state_code2,...,covered_recipient_license_state_code4,covered_recipient_license_state_code5,recipient_primary_business_street_address_line1,recipient_primary_business_street_address_line2,recipient_city,recipient_state,recipient_zip_code,recipient_country,recipient_postal_code,program_year
0,294069,1497734156,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,None,NC,NaN,...,NaN,NaN,2001 VAIL AVE,SUITE 200B,CHARLOTTE,NC,28207,United States,NaN,2023
4090128,230945,1104092055,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,None,MA,NaN,...,NaN,NaN,725 ALBANY STREET,"SHAPIRO 4, SUITE B",BOSTON,MA,02118,United States,NaN,2023
4090100,982045,1720244262,Covered Recipient Physician,Doctor of Osteopathy,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,None,ME,NaN,...,NaN,NaN,78 RIDGEWOOD DR,NaN,BANGOR,ME,04401,United States,NaN,2023
4090101,48541,1063670065,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,None,MA,NaN,...,NaN,NaN,300 HALKET ST,SUITE 1601B,PITTSBURGH,PA,15213,United States,NaN,2023
4090102,113060,1497043806,Covered Recipient Physician,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers...,Podiatric Medicine & Surgery Service Providers,Podiatrist,Foot & Ankle Surgery,KY,NaN,...,NaN,NaN,550 S JACKSON ST,NaN,LOUISVILLE,KY,40202,United States,NaN,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4975771,15112,1336171040,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Internal M...,Allopathic & Osteopathic Physicians,Internal Medicine,Pulmonary Disease,NY,NaN,...,NaN,NaN,37-11 88TH STREET,NaN,JACKSON HEIGHTS,NY,11372,United States,NaN,2021
4975772,777099,1538182779,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Internal M...,Allopathic & Osteopathic Physicians,Internal Medicine,Pulmonary Disease,CO,NaN,...,NaN,NaN,5651 FIRST BLVD,STE 408,HERMITAGE,TN,37076,United States,NaN,2021
4975773,498662,1740473990,Covered Recipient Physician,Doctor of Dentistry,Dental Providers|Dentist|General Practice,Dental Providers,Dentist,General Practice,WA,NaN,...,NaN,NaN,1000 WALLACE WAY,NaN,GRANDVIEW,WA,98930,United States,NaN,2021
4975774,8800040,1699323576,Covered Recipient Physician,Doctor of Dentistry,Dental Providers|Dentist,Dental Providers,Dentist,None,NY,NaN,...,NaN,NaN,329 S MAIN ST,NaN,CANANDAIGUA,NY,14424,United States,NaN,2021


In [28]:
# KEEP ONLY MOST RECENT PROVIDER INFORMATION
providers = providers.drop_duplicates(keep='first').copy()

In [29]:
providers.to_csv("/dsa/groups/casestudycf25/team02/general_payments_providers_clean.csv")

In [30]:
del providers

########################################
# CREATE PRODUCTS DIM
########################################

In [31]:
#  1 to 5 cols
slots = range(1, 6)

long_frames = []

for i in slots:
    tmp = df[[
        'record_id',
        f'associated_device_or_medical_supply_pdi_{i}',
        f'name_of_drug_or_biological_or_device_or_medical_supply_{i}',
        f'covered_or_noncovered_indicator_{i}',
        f'product_category_or_therapeutic_area_{i}',
        f'indicate_drug_or_biological_or_device_or_medical_supply_{i}'
    ]].copy()

    tmp.columns = [
        'record_id',
        'pdi',
        'product_name',
        'covered_indicator',
        'product_category',
        'drug_or_device_indicator'
    ]

    # Keep the source slot for debugging or tracking
    tmp['slot'] = i

    long_frames.append(tmp)

# Concatenate into single long table
products = pd.concat(long_frames, ignore_index=True)

# Drop empty rows where no product was specified
products = products.dropna(subset=['pdi', 'product_name'], how='all')

products

,record_id,pdi,product_name,covered_indicator,product_category,drug_or_device_indicator,slot
0,1006722961,10859477002140,AUGMENT INJECTABLE,Covered,BONE SUBSTITUTES,Device,1
1,1006722977,00889797103404,EASYFUSE,Covered,TRAUMA & EXTREMITIES,Device,1
2,1006723021,00840420121912,INFINITY,Covered,TRAUMA & EXTREMITIES,Device,1
3,1006723073,00840420123022,PRO-TOE,Covered,TRAUMA & EXTREMITIES,Device,1
4,1006530267,10888857031555,CAPRI CORPECTOMY CAGE SYSTEM,Covered,SPINE,Device,1
...,...,...,...,...,...,...,...
32810473,887081995,15712123000318,SURGIFLO HEMOSTATIC MATRIX,Covered,BIOSURGICAL,Device,5
32811076,990843001,10705036014171,ENSEAL,Covered,ENERGY,Device,5
32811182,887071747,10705036014928,ECHELON ENDOPATH,Covered,ENDO-MECHANICAL,Device,5
32811543,843219165,NaN,STRAVIX,Covered,EXTREMITIES & LIMB RESTORATION,Medical Supply,5


In [32]:
# get a df of product names and pdis: looks like [id, [name1,name2,3]]
names_list_product_ids = (
    products.groupby('pdi')['product_name']
           .apply(lambda s: s.dropna().unique())
           .to_dict()
)

# map names to a list of ids and only apply it if there is a single valid id
single_pdi_map = {
    name: vals[0]
    for name, vals in names_list_product_ids.items()
    if len(vals) == 1
}

# Create new temp column 
products['name_filled'] = products['pdi'].map(single_pdi_map)

In [33]:
# if pdi is null then fill with the new pdi_filled colum
products['product_name'] = products['product_name'].fillna(products['name_filled'])

In [34]:
# drop the temp column
products = products.drop(columns=['name_filled'])

In [35]:
products.shape

(7982139, 7)

In [36]:
products[
#     products['pdi'].isna() &
    products['product_name'].isna()
]

,record_id,pdi,product_name,covered_indicator,product_category,drug_or_device_indicator,slot
38,1006812967,07613327129243,NaN,Covered,NEURO,Device,1
47,1006793685,04546540355355,NaN,Covered,CRANIOMAXILLOFACIAL,Device,1
63,1006640065,07613327096835,NaN,Covered,NEURO SPINE ENT,Device,1
96,1006760517,04546540355355,NaN,Covered,CRANIOMAXILLOFACIAL,Device,1
103,1006806399,07613327096439,NaN,Covered,NEURO SPINE ENT,Device,1
...,...,...,...,...,...,...,...
32528388,806453619,04546540365415,NaN,Covered,CRANIOMAXILLOFACIAL,Device,5
32532246,787476767,10886982187017,NaN,Covered,TRAUMA,Device,5
32532864,787496865,10886982231215,NaN,Covered,TRAUMA,Device,5
32533951,787584395,10705036003465,NaN,Covered,ENDO-MECHANICAL,Device,5


In [37]:
# Every column can still be nullable except for record id
products.to_csv("/dsa/groups/casestudycf25/team02/general_payments_products_clean.csv")

In [38]:
del products

########################################
# CREATE Manufacturers DIM
########################################

In [39]:

manufacturers = df[[
"applicable_manufacturer_or_applicable_gpo_making_payment_id",
"applicable_manufacturer_or_applicable_gpo_making_payment_name",
"applicable_manufacturer_or_applicable_gpo_making_payment_state",
"applicable_manufacturer_or_applicable_gpo_making_payment_country", "program_year"]]

In [40]:
manufacturers.sort_values(by = "program_year", ascending = False)

,applicable_manufacturer_or_applicable_gpo_making_payment_id,applicable_manufacturer_or_applicable_gpo_making_payment_name,applicable_manufacturer_or_applicable_gpo_making_payment_state,applicable_manufacturer_or_applicable_gpo_making_payment_country,program_year
0,100000010503,STRYKER CORPORATION,MI,United States,2023
4090128,100000005456,SMITH+NEPHEW INC,TN,United States,2023
4090100,100000005456,SMITH+NEPHEW INC,TN,United States,2023
4090101,100000005456,SMITH+NEPHEW INC,TN,United States,2023
4090102,100000005456,SMITH+NEPHEW INC,TN,United States,2023
...,...,...,...,...,...
4975771,100000005391,ADVANCED RESPIRATORY INC,MN,United States,2021
4975772,100000005391,ADVANCED RESPIRATORY INC,MN,United States,2021
4975773,100000005613,ALIGN TECHNOLOGY INC,AZ,United States,2021
4975774,100000005613,ALIGN TECHNOLOGY INC,AZ,United States,2021


In [41]:
manufacturers = df[[
"applicable_manufacturer_or_applicable_gpo_making_payment_id",
"applicable_manufacturer_or_applicable_gpo_making_payment_name",
"applicable_manufacturer_or_applicable_gpo_making_payment_state",
"applicable_manufacturer_or_applicable_gpo_making_payment_country"]]

In [42]:
manufacturers = manufacturers.drop_duplicates(keep='first').copy()

In [43]:
# Drop duplicates based on manufacturing id
manufacturers = manufacturers.drop_duplicates(keep='first',
                                              subset= "applicable_manufacturer_or_applicable_gpo_making_payment_id").copy()

In [44]:
print("Check if there are any duplicate record ids in final dataset: ")
manufacturers["applicable_manufacturer_or_applicable_gpo_making_payment_id"].duplicated().any()

Check if there are any duplicate record ids in final dataset: 


False

In [45]:
manufacturers.to_csv("/dsa/groups/casestudycf25/team02/general_payments_manufacturers_clean.csv")

In [46]:
del manufacturers

########################################
# CREATE REMAINING FACT TAB
########################################

This has the General Payments leftover data at the record level

In [47]:
sliced_df = df[[ 'record_id', 'change_type',  'payment_publication_date', 'date_of_payment', 'program_year', 
       'covered_recipient_npi',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'total_amount_of_payment_us_dollars', 
       'number_of_payments_included_in_total_amount',
       'form_of_payment_or_transfer_of_value',
       'nature_of_payment_or_transfer_of_value', 'nature_short_descr', 
       'city_of_travel',
       'state_of_travel', 'country_of_travel', 
        'physician_ownership_indicator',
       'third_party_payment_recipient_indicator',
       'name_of_third_party_entity_receiving_payment_or_transfer_of_value',
       'charity_indicator', 'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication'
     ]]

In [48]:
sliced_df.to_csv("/dsa/groups/casestudycf25/team02/general_payments-sliced_clean.csv")

In [49]:
del sliced_df

In [50]:
col_totals = df.select_dtypes(include="number").sum()
print("All Column Totals")
print("-" *50)
print(col_totals.apply(lambda x: f"{x:.0f}"))

All Column Totals
--------------------------------------------------
covered_recipient_npi                          9839652527502438
total_amount_of_payment_us_dollars                   3109903239
number_of_payments_included_in_total_amount             6660847
program_year                                        13270896440
dtype: object


##############################################################
# Return the intact and large dataset as zipped CSV in chunks
##############################################################

This dataset is before the dimensions splitting but should be cleaned the same way.

In [51]:
# df_filtered.to_csv("/dsa/groups/casestudycf25/team02/cms_general_payments_clean.csv.gz", index=False, compression="gzip")
# df_filtered.to_csv("cms_general_payments_clean.csv.gz", index=False, compression="gzip")

import math

rows_per_file = 1_000_000
n_rows = len(df)
n_files = math.ceil(n_rows / rows_per_file)

for i in range(n_files):
    start = i * rows_per_file
    end = start + rows_per_file

    outname = f"cms_general_payments_clean_part_{i+1}.csv.gz"
    df.iloc[start:end].to_csv(outname, index=False, compression="gzip")

    print(f"Created {outname}")
df.shape

Created cms_general_payments_clean_part_1.csv.gz
Created cms_general_payments_clean_part_2.csv.gz
Created cms_general_payments_clean_part_3.csv.gz
Created cms_general_payments_clean_part_4.csv.gz
Created cms_general_payments_clean_part_5.csv.gz
Created cms_general_payments_clean_part_6.csv.gz
Created cms_general_payments_clean_part_7.csv.gz


(6562870, 68)

In [52]:
# Run locally
# import pandas as pd
# import glob

# # Adjust the path/pattern to match your files
# files = glob.glob("raw_data/cms_general_payments_cleaned/cms_general_payments_clean_part_*.csv")

# df_list = []

# for f in files:
#     print("Reading:", f)
#     df_list.append(pd.read_csv(f, low_memory=False))

# # Combine all into one big dataframe
# df = pd.concat(df_list, ignore_index=True)